# CPRI Hackathon — Task 01: Person 2
## P2-Stage 4: Final Validity Model & Inference Pipeline

This notebook demonstrates the **final Stage 4 Valid vs Invalid classifier** pipeline for Task 01.

### Key Highlights:
1. **Leakage-Free 5-Fold Stratified CV**: Evaluates candidate models (Logistic Regression, Random Forest, Gradient Boosting).
2. **Threshold Optimization**: Decision threshold tuned on OOF probabilities to maximize **Invalid F1**.
3. **Person 1 Adapter**: Clean fallback handling for regime/consistency features.
4. **Final Inference API**: Predicts on `Test_Data` (350 records) without requiring `Validity_Label` or `Reference_Parameter`.

In [ ]:
import os, sys, pandas as pd, numpy as np
sys.path.insert(0, '../src')
from stage4_features import load_stage4_datasets, prepare_stage4_feature_sets
from stage4_models import evaluate_stage4_cv
from final_validity_model import FinalValidityClassifier

dataset_path = '../data/CPRI_Hackathon_Screening_Dataset_PARTICIPANT.xlsx'
df_tr_flagged, df_te_flagged = load_stage4_datasets(dataset_path)
print(f'Training_Data: {len(df_tr_flagged)} records, Test_Data: {len(df_te_flagged)} records.')

In [ ]:
# Candidate Feature Sets & Constant Feature Analysis
prep = prepare_stage4_feature_sets(df_tr_flagged, df_te_flagged)
print(f'Genuine Person 1 features present: {prep["is_genuine_person1"]}')
print(f'Candidate Feature Set Sizes: Set A: {len(prep["feature_set_cols"]["Set_A"])}, Set B: {len(prep["feature_set_cols"]["Set_B"])}, Set C: {len(prep["feature_set_cols"]["Set_C"])}')

In [ ]:
# Final Classifier Fit & Test_Data Inference
clf = FinalValidityClassifier(model_name='random_forest', feature_set_name='Set_C', decision_threshold=0.45, random_state=42)
clf.fit(df_tr_flagged)

test_preds = clf.predict_dataset(df_te_flagged)
print(f'Test Data Predictions Head:')
test_preds.head(10)